Absolute Positional Embedding

In [7]:
import torch
import tiktoken
from torch.utils.data import DataLoader, Dataset

In [ ]:
#total number of unique tokens that LLM can represent
vocab_size = 50257

#how many dimensions each token can be represented by (vector)
output_dim = 256

#Create embedding layer
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [9]:
#create a dataset class that takes in the text, tokenizer, max_length, and stride
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids =[]
        token_ids = tokenizer.encode(txt, allowed_special = {"<|endoftext|>"}) 

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
#Batch size = num of input sequence
#Context size = tokens per input sequence = max_length

def create_dataLoader_V1(txt, batch_size = 4, max_length = 256, stride = 128, shuffle = True, drop_last = True, num_workers = 0):

    #tokenizes using BPE
    tokenizer = tiktoken.get_encoding("gpt2")

    #creates a dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    #creates a DataLoader
    data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

    return data_loader

In [11]:
#read the text file
with open("the-verdict.txt", "r", encoding = "utf-8") as f:
    raw_text = f.read()

In [ ]:
#context size
max_length = 4

dataloader = create_dataLoader_V1(raw_text,batch_size = 8, max_length = max_length, stride = max_length, shuffle = False)

data_iter = iter(dataloader)

inputs,targets = next(data_iter)